# 🧠 Neural Networks & How They Learn: An Interactive Visual Guide

Welcome! This interactive notebook explores the mathematics, visual intuition, and implementation behind **Artificial Neural Networks** and **Backpropagation**, blending animated geometric visualizations with pure NumPy implementations and hands-on digit classification.

---

### 🗺️ Table of Contents
1. **[Setup & Dependencies](#setup)**
2. **[Chapter 1: Anatomy of a Neural Network](#chapter1)**
   - Neurons as Activations ($a \in [0, 1]$)
   - The Layer Architecture ($784 \to 16 \to 16 \to 10$)
   - Synaptic Weights, Biases & Activation Functions (Sigmoid & ReLU)
   - *🎬 Manim Animation 1: Multi-Layer Perceptron Architecture*
   - *🎬 Manim Animation 2: Sigmoid Activation Function*
3. **[Chapter 2: The Cost Function & Gradient Descent](#chapter2)**
   - Quantifying Error: The Mean Squared Error (MSE) Loss
   - The High-Dimensional Optimization Landscape
   - The Gradient Vector $\nabla C$ & Stepping Downhill
   - *🎬 Manim Animation 3: Gradient Descent on the Cost Landscape*
4. **[Chapter 3: Backpropagation & The Chain Rule](#chapter3)**
   - What is Backpropagation Really Doing?
   - The Calculus of a Single Synaptic Connection
   - Error Propagation across Layers (Vectorized Form)
   - Mini-Batch Stochastic Gradient Descent (SGD)
   - *🎬 Manim Animation 4: Backpropagation Signal Flow*
5. **[Chapter 4: Pure NumPy Neural Network Implementation](#chapter4)**
   - Building the `Network` class from scratch
6. **[Chapter 5: Training & Live Digit Classification](#chapter5)**
   - Training on MNIST Digits
   - Loading Pretrained Weights & Real-time Digit Recognition

---
<a id="setup"></a>
## ⚙️ 1. Setup & Environment

We use **Manim Community Edition** (`manim`) to render animations directly inside this Jupyter notebook. We also configure low-latency inline video previews (`-qm` for 720p medium quality, or `-ql` for 480p fast rendering).

In [1]:
import os
import sys
import pickle
import numpy as np
import manim
from manim import *

# Ensure local module directory is in sys.path
if os.path.abspath("_2017") not in sys.path:
    sys.path.append(os.path.abspath("_2017"))

# Configure Manim notebook video embedding
config.media_embed = True
config.media_width = "85%"
config.verbosity = "WARNING"

print(f"Manim Community Version: {manim.__version__}")
print(f"NumPy Version: {np.__version__}")
print("Environment successfully configured!")

Manim Community Version: 0.21.0
NumPy Version: 2.5.3
Environment successfully configured!


---
<a id="chapter1"></a>
## 🧱 2. Chapter 1: Anatomy of a Neural Network

### What is a Neuron?
In machine learning, a **neuron** is simply a container that holds a number—specifically an **activation** $a \in [0, 1]$.

For hand-written digit recognition on the famous **MNIST** dataset:
- **Input Layer ($a^{(0)}$)**: $28 \times 28 = 784$ neurons, where each neuron represents the grayscale brightness of a pixel ($0.0$ for black, $1.0$ for white).
- **Hidden Layers ($a^{(1)}, a^{(2)}$)**: Intermediate representations. Intuitively, these detect edges, loops, and stroke combinations.
- **Output Layer ($a^{(L)}$)**: $10$ neurons representing the digits $0, 1, 2, \dots, 9$. The neuron with the highest activation indicates the network's prediction.

```
Input (784 pixels) ───► Hidden Layer 1 (16) ───► Hidden Layer 2 (16) ───► Output (10 classes)
```

### The Linear Combination & Bias
To compute the activation of neuron $j$ in layer $l$, we take the weighted sum of all activations in the previous layer, add a bias $b_j^{(l)}$, and pass the result through an activation function $\sigma$:

$$z_j^{(l)} = \sum_k w_{jk}^{(l)} a_k^{(l-1)} + b_j^{(l)}$$

$$a_j^{(l)} = \sigma\left( z_j^{(l)} \right)$$

In compact matrix-vector notation:
$$\mathbf{a}^{(l)} = \sigma\left( \mathbf{W}^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)} \right)$$

- $\mathbf{W}^{(l)}$: Weight matrix where row $j$, column $k$ is the connection weight from neuron $k$ in layer $l-1$ to neuron $j$ in layer $l$.
- $\mathbf{b}^{(l)}$: Bias vector adjusting the threshold for activation.
- $\sigma$: Non-linear activation function.

In [2]:
%%manim -qm -v WARNING NetworkArchitectureScene

class NetworkArchitectureScene(Scene):
    def construct(self):
        title = Title("Neural Network Architecture", font_size=38)
        self.play(Write(title))
        
        # Define layer sizes for visual clarity: [4 inputs, 6 hidden, 6 hidden, 3 outputs]
        layer_sizes = [4, 6, 6, 3]
        layer_names = ["Input a^(0)\n(784)", "Hidden a^(1)\n(16)", "Hidden a^(2)\n(16)", "Output a^(3)\n(10)"]
        layers = VGroup()
        
        x_spacing = 2.4
        for l_idx, size in enumerate(layer_sizes):
            layer = VGroup()
            y_spacing = 0.65
            y_start = (size - 1) * y_spacing / 2
            x_pos = (l_idx - 1.5) * x_spacing
            
            for n_idx in range(size):
                y_pos = y_start - n_idx * y_spacing
                neuron = Circle(radius=0.22, color=BLUE_D, fill_opacity=0.3, fill_color=BLUE_E)
                neuron.move_to([x_pos, y_pos - 0.2, 0])
                layer.add(neuron)
            layers.add(layer)
            
        # Draw synaptic connections
        synapses = VGroup()
        for l in range(len(layers) - 1):
            for n1 in layers[l]:
                for n2 in layers[l+1]:
                    line = Line(n1.get_right(), n2.get_left(), stroke_width=0.8, color=GRAY_C, stroke_opacity=0.35)
                    synapses.add(line)
                    
        self.play(FadeIn(layers), run_time=1.0)
        self.play(Create(synapses, lag_ratio=0.005), run_time=1.5)
        
        # Light up activations through forward propagation
        activations = [
            [0.9, 0.2, 0.8, 0.1],
            [0.7, 0.9, 0.1, 0.8, 0.4, 0.6],
            [0.8, 0.3, 0.95, 0.2, 0.7, 0.1],
            [0.05, 0.98, 0.02]
        ]
        
        for l_idx, layer in enumerate(layers):
            anims = [neuron.animate.set_fill(YELLOW, opacity=activations[l_idx][n_idx]) 
                     for n_idx, neuron in enumerate(layer)]
            self.play(*anims, run_time=0.5)
            
        # Formula
        formula = MathTex(r"\mathbf{a}^{(l)} = \sigma\left( \mathbf{W}^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)} \right)", font_size=36)
        formula.to_edge(DOWN)
        formula.set_color(YELLOW)
        self.play(Write(formula))
        self.wait(1.5)

Manim Community v0.21.0

### The Activation Function $\sigma(z)$
Without a non-linear activation function, a deep neural network is just a composition of linear transformations—which collapses into a single linear transformation!

1. **Sigmoid $\sigma(z)$**:
   $$\sigma(z) = \frac{1}{1 + e^{-z}}$$
   - Maps $(-\infty, \infty) \to (0, 1)$, smoothly squishing very negative values toward $0$ and very positive values toward $1$.
   - Derivative has an elegant algebraic property: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$.

2. **ReLU (Rectified Linear Unit)**:
   $$\text{ReLU}(z) = \max(0, z)$$
   - Popular in modern architectures to avoid vanishing gradients.

In [3]:
%%manim -qm -v WARNING ActivationFunctionsScene

class ActivationFunctionsScene(Scene):
    def construct(self):
        title = Title("Activation Function: The Sigmoid Curve", font_size=38)
        self.play(Write(title))
        
        ax = Axes(
            x_range=[-6, 6, 2],
            y_range=[-0.2, 1.2, 0.5],
            x_length=7.5,
            y_length=4.0,
            axis_config={"color": BLUE},
        ).shift(DOWN * 0.3)
        
        labels = ax.get_axis_labels(x_label="z", y_label=r"\sigma(z)")
        
        sigmoid_graph = ax.plot(lambda x: 1 / (1 + np.exp(-x)), color=YELLOW)
        sigmoid_label = MathTex(r"\sigma(z) = \frac{1}{1 + e^{-z}}", color=YELLOW, font_size=34).next_to(ax, UP)
        
        self.play(Create(ax), Write(labels))
        self.play(Create(sigmoid_graph), Write(sigmoid_label), run_time=1.5)
        
        # Interactive point demonstrating squishing
        dot = Dot(color=RED, radius=0.1).move_to(ax.c2p(0, 0.5))
        dot_label = MathTex(r"(0, 0.5)", font_size=24, color=RED).next_to(dot, UR, buff=0.1)
        
        self.play(FadeIn(dot), Write(dot_label))
        
        # Move right
        self.play(
            dot.animate.move_to(ax.c2p(3.5, 1 / (1 + np.exp(-3.5)))),
            dot_label.animate.move_to(ax.c2p(3.5, 1 / (1 + np.exp(-3.5))) + UR * 0.3),
            run_time=1.2
        )
        
        # Move left
        self.play(
            dot.animate.move_to(ax.c2p(-3.5, 1 / (1 + np.exp(3.5)))),
            dot_label.animate.move_to(ax.c2p(-3.5, 1 / (1 + np.exp(3.5))) + UR * 0.3),
            run_time=1.2
        )
        self.wait(1)

Manim Community v0.21.0

---
<a id="chapter2"></a>
## 📉 3. Chapter 2: The Cost Function & Gradient Descent

### Quantifying "How Bad" the Network Is
When an image of a **$3$** is passed into an untrained network with random weights, the output activations might look like garbage:
$$a^{(L)} = [0.42, 0.81, 0.12, 0.05, \dots, 0.77]^T$$
Whereas the target output $y$ is a 1-hot vector with a $1$ at index 3:
$$y = [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]^T$$

To quantify total error, we define the **Cost Function** (Mean Squared Error):
$$C(W, b) = \frac{1}{2n} \sum_{x} \| y(x) - a^{(L)}(x) \|^2$$

For a single training example $x$:
$$C_0 = \frac{1}{2} \sum_{j=0}^{9} \left( a_j^{(L)} - y_j \right)^2$$

### The Optimization Landscape: Stepping Downhill
In a network with $784 \to 16 \to 16 \to 10$, there are:
$$(784 \times 16 + 16) + (16 \times 16 + 16) + (16 \times 10 + 10) = 13{,}002 \text{ parameters!}$$

The cost $C(\theta)$ is a function in a $13{,}002$-dimensional space. The **gradient** $\nabla C$ is a vector of partial derivatives pointing in the direction of steepest *increase*:

$$\nabla C = \begin{bmatrix} \frac{\partial C}{\partial w_1} \\ \frac{\partial C}{\partial w_2} \\ \vdots \\ \frac{\partial C}{\partial b_k} \end{bmatrix}$$

To minimize the cost, we take steps in the direction of the **negative gradient** $-\nabla C$:

$$\theta \leftarrow \theta - \eta \nabla C$$

where $\eta > 0$ is the **learning rate**.

In [4]:
%%manim -qm -v WARNING GradientDescentScene

class GradientDescentScene(Scene):
    def construct(self):
        title = Title("Gradient Descent on the Cost Landscape", font_size=38)
        self.play(Write(title))
        
        ax = Axes(
            x_range=[-3, 3, 1],
            y_range=[0, 9, 2],
            x_length=7.5,
            y_length=4.2,
            axis_config={"color": BLUE},
        ).shift(DOWN * 0.4)
        
        labels = ax.get_axis_labels(x_label="w", y_label="C(w)")
        cost_curve = ax.plot(lambda x: x**2, color=YELLOW)
        cost_eq = MathTex(r"C(w) = w^2", color=YELLOW, font_size=32).next_to(ax, UP)
        
        self.play(Create(ax), Write(labels), Create(cost_curve), Write(cost_eq))
        
        # Initial point
        w_val = 2.5
        dot = Dot(ax.c2p(w_val, w_val**2), color=RED, radius=0.12)
        step_text = MathTex(r"w \leftarrow w - \eta \frac{dC}{dw}", font_size=32, color=GREEN).to_corner(UR).shift(DOWN*0.5+LEFT*0.3)
        
        self.play(FadeIn(dot), Write(step_text))
        
        # Take gradient descent steps
        eta = 0.35
        for step in range(4):
            grad = 2 * w_val
            w_next = w_val - eta * grad
            arrow = Arrow(
                ax.c2p(w_val, w_val**2), 
                ax.c2p(w_next, w_next**2), 
                buff=0, 
                color=GREEN, 
                stroke_width=3.5,
                max_tip_length_to_length_ratio=0.25
            )
            self.play(Create(arrow), dot.animate.move_to(ax.c2p(w_next, w_next**2)), run_time=0.7)
            w_val = w_next
        self.wait(1)

Manim Community v0.21.0

---
<a id="chapter3"></a>
## 🔄 4. Chapter 3: Backpropagation & The Chain Rule

### How Do We Efficiently Compute $\nabla C$?
Computing partial derivatives numerically by nudging each of the $13{,}000$ weights one-by-one would require $13{,}000$ forward passes per training sample—prohibitively slow!

**Backpropagation** computes all $13{,}000$ partial derivatives in a single backward pass using the **Chain Rule of Calculus**.

### Chain Rule for a Single Synapse
Consider how a weight $w^{(l)}_{jk}$ affects the final cost $C_0$:
$$w^{(l)}_{jk} \xrightarrow{\text{linear combo}} z^{(l)}_j \xrightarrow{\text{activation } \sigma} a^{(l)}_j \xrightarrow{\dots} C_0$$

Applying the chain rule:
$$\frac{\partial C_0}{\partial w^{(l)}_{jk}} = \frac{\partial z^{(l)}_j}{\partial w^{(l)}_{jk}} \cdot \frac{\partial a^{(l)}_j}{\partial z^{(l)}_j} \cdot \frac{\partial C_0}{\partial a^{(l)}_j}$$

Let's evaluate each factor:
1. Since $z^{(l)}_j = \sum_k w^{(l)}_{jk} a^{(l-1)}_k + b^{(l)}_j$, we have:
   $$\frac{\partial z^{(l)}_j}{\partial w^{(l)}_{jk}} = a^{(l-1)}_k$$
2. Since $a^{(l)}_j = \sigma(z^{(l)}_j)$, we have:
   $$\frac{\partial a^{(l)}_j}{\partial z^{(l)}_j} = \sigma'(z^{(l)}_j)$$
3. For the output layer where $C_0 = \frac{1}{2} \sum (a^{(L)}_j - y_j)^2$:
   $$\frac{\partial C_0}{\partial a^{(L)}_j} = (a^{(L)}_j - y_j)$$

Multiplying them together gives the exact gradient:
$$\frac{\partial C_0}{\partial w^{(L)}_{jk}} = a^{(L-1)}_k \cdot \sigma'(z^{(L)}_j) \cdot (a^{(L)}_j - y_j)$$

Similarly for the bias:
$$\frac{\partial C_0}{\partial b^{(L)}_j} = \sigma'(z^{(L)}_j) \cdot (a^{(L)}_j - y_j)$$

### The 4 Fundamental Equations of Backpropagation
Defining the error $\delta^{(l)}_j \equiv \frac{\partial C}{\partial z^{(l)}_j}$:

1. **Output Error**:
   $$\boldsymbol{\delta}^{(L)} = \nabla_{\mathbf{a}} C \odot \sigma'(\mathbf{z}^{(L)}) = (\mathbf{a}^{(L)} - \mathbf{y}) \odot \sigma'(\mathbf{z}^{(L)})$$

2. **Error Propagation to Previous Layer**:
   $$\boldsymbol{\delta}^{(l)} = \left( (\mathbf{W}^{(l+1)})^T \boldsymbol{\delta}^{(l+1)} \right) \odot \sigma'(\mathbf{z}^{(l)})$$

3. **Rate of Change for Biases**:
   $$\frac{\partial C}{\partial b^{(l)}_j} = \delta^{(l)}_j$$

4. **Rate of Change for Weights**:
   $$\frac{\partial C}{\partial w^{(l)}_{jk}} = a^{(l-1)}_k \delta^{(l)}_j \implies \frac{\partial C}{\partial \mathbf{W}^{(l)}} = \boldsymbol{\delta}^{(l)} (\mathbf{a}^{(l-1)})^T$$

In [5]:
%%manim -qm -v WARNING BackpropagationScene

class BackpropagationScene(Scene):
    def construct(self):
        title = Title("Backpropagation: The Chain Rule in Action", font_size=38)
        self.play(Write(title))
        
        # 3 computational nodes: a^(l-1), a^(l), Cost C
        n_prev = Circle(radius=0.38, color=BLUE, fill_opacity=0.3, fill_color=BLUE).shift(LEFT * 4.2 + UP * 0.5)
        n_curr = Circle(radius=0.38, color=YELLOW, fill_opacity=0.3, fill_color=YELLOW).shift(LEFT * 0.5 + UP * 0.5)
        n_cost = Square(side_length=0.76, color=RED, fill_opacity=0.3, fill_color=RED).shift(RIGHT * 3.8 + UP * 0.5)
        
        lbl_prev = MathTex(r"a^{(l-1)}", font_size=32).next_to(n_prev, DOWN)
        lbl_curr = MathTex(r"a^{(l)} = \sigma(z^{(l)})", font_size=30).next_to(n_curr, DOWN)
        lbl_cost = MathTex(r"C = \frac{1}{2}(a - y)^2", font_size=30).next_to(n_cost, DOWN)
        
        edge1 = Arrow(n_prev.get_right(), n_curr.get_left(), buff=0.1, color=WHITE)
        edge1_lbl = MathTex(r"w^{(l)}, b^{(l)}", font_size=26).next_to(edge1, UP)
        
        edge2 = Arrow(n_curr.get_right(), n_cost.get_left(), buff=0.1, color=WHITE)
        edge2_lbl = MathTex(r"\text{error}", font_size=26).next_to(edge2, UP)
        
        self.play(FadeIn(n_prev), FadeIn(n_curr), FadeIn(n_cost),
                  Write(lbl_prev), Write(lbl_curr), Write(lbl_cost),
                  Create(edge1), Write(edge1_lbl),
                  Create(edge2), Write(edge2_lbl))
        
        # Forward pass signal
        pulse = Dot(color=YELLOW, radius=0.12).move_to(n_prev.get_center())
        self.play(FadeIn(pulse))
        self.play(pulse.animate.move_to(n_curr.get_center()), run_time=0.6)
        self.play(pulse.animate.move_to(n_cost.get_center()), run_time=0.6)
        self.play(FadeOut(pulse))
        
        # Backprop error pulse (backward arrow)
        back_arrow = Arrow(n_cost.get_left(), n_curr.get_right(), color=RED, buff=0.1)
        self.play(Create(back_arrow))
        
        # Chain rule equation
        chain_eq = MathTex(
            r"\frac{\partial C}{\partial w^{(l)}} = "
            r"\overbrace{\frac{\partial z^{(l)}}{\partial w^{(l)}}}^{a^{(l-1)}} \cdot "
            r"\overbrace{\frac{\partial a^{(l)}}{\partial z^{(l)}}}^{\sigma^\prime(z^{(l)})} \cdot "
            r"\overbrace{\frac{\partial C}{\partial a^{(l)}}}^{(a^{(l)} - y)}",
            font_size=34
        ).shift(DOWN * 2.2)
        chain_eq.set_color_by_tex(r"\frac{\partial C}{\partial w^{(l)}}", RED)
        
        self.play(Write(chain_eq), run_time=2.0)
        self.wait(1.5)

Manim Community v0.21.0

---
<a id="chapter4"></a>
## 💻 5. Chapter 4: Pure NumPy Neural Network Implementation

Here is a clean, fully-vectorized Python implementation of the feedforward neural network and backpropagation algorithm using pure NumPy.

In [6]:
import random
import numpy as np

def sigmoid(z):
    """Numerically stable sigmoid activation function."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_prime(z):
    """Derivative of the sigmoid function: sigma'(z) = sigma(z) * (1 - sigma(z))."""
    s = sigmoid(z)
    return s * (1.0 - s)

class Network:
    def __init__(self, sizes):
        """
        The list `sizes` contains the number of neurons in each layer.
        Example: [784, 16, 16, 10] initializes a 4-layer network.
        """
        self.num_layers = len(sizes)
        self.sizes = sizes
        # Initialize biases and weights randomly from standard normal distribution
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x) / np.sqrt(x) 
                        for x, y in zip(sizes[:-1], sizes[1:])]

    def feedforward(self, a):
        """Return the output activations of the network for input vector a."""
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a) + b)
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta, test_data=None):
        """
        Train the neural network using Mini-Batch Stochastic Gradient Descent.
        - training_data: list of tuples (x, y)
        - eta: learning rate
        """
        n = len(training_data)
        history = {'epochs': [], 'accuracy': []}
        
        for j in range(epochs):
            random.shuffle(training_data)
            mini_batches = [
                training_data[k:k+mini_batch_size]
                for k in range(0, n, mini_batch_size)
            ]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)
                
            if test_data:
                n_test = len(test_data)
                correct = self.evaluate(test_data)
                acc = (correct / n_test) * 100
                history['epochs'].append(j + 1)
                history['accuracy'].append(acc)
                print(f"Epoch {j+1:2d} / {epochs}: {correct} / {n_test} ({acc:.2f}%) correct")
            else:
                print(f"Epoch {j+1:2d} / {epochs} complete")
                
        return history

    def update_mini_batch(self, mini_batch, eta):
        """Update weights and biases by applying gradient descent using backpropagation on a mini batch."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        
        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb + dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw + dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
            
        # Parameter update step: theta <- theta - (eta / m) * sum(delta_theta)
        self.weights = [w - (eta / len(mini_batch)) * nw 
                        for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b - (eta / len(mini_batch)) * nb 
                       for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x, y):
        """Return a tuple (nabla_b, nabla_w) representing the gradient for a single training example (x, y)."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        
        # 1. Forward Pass
        activation = x
        activations = [x] # list of layer activations
        zs = []           # list of z vectors
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
            
        # 2. Backward Pass
        # Output layer error: delta^L = (a^L - y) * sigma'(z^L)
        delta = self.cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1])
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].T)
        
        # Propagate error backward through hidden layers
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            # delta^l = ((W^(l+1))^T delta^(l+1)) * sigma'(z^l)
            delta = np.dot(self.weights[-l+1].T, delta) * sp
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l-1].T)
            
        return (nabla_b, nabla_w)

    def evaluate(self, test_data):
        """Return the number of test inputs for which the neural network outputs the correct result."""
        test_results = [(np.argmax(self.feedforward(x)), y if isinstance(y, (int, np.integer)) else np.argmax(y))
                        for (x, y) in test_data]
        return sum(int(pred == target) for (pred, target) in test_results)

    def cost_derivative(self, output_activations, y):
        """Return the vector of partial derivatives dC / da for the output activations."""
        return (output_activations - y)

print("Network class successfully compiled!")

Network class successfully compiled!


---
<a id="chapter5"></a>
## 🎯 6. Chapter 5: Training & Live Digit Classification

Let's test our network on real MNIST digit images! 

We can:
1. Load sample digit images directly from the dataset (`./_2017/nn/image_map`).
2. Train a fresh network from scratch and watch accuracy climb.
3. Load pretrained network weights (`./_2017/nn/pretrained_weights_and_biases`) to inspect high-confidence predictions.

In [7]:
# Load Sample MNIST Images
import os
import pickle
import random
import numpy as np

image_map_path = os.path.join("_2017", "nn", "image_map")
pretrained_path = os.path.join("_2017", "nn", "pretrained_weights_and_biases")

with open(image_map_path, "rb") as f:
    image_map = pickle.load(f, encoding="latin1")

print(f"Loaded sample images for digits: {list(image_map.keys())}")
print(f"Number of samples per digit: {[len(image_map[d]) for d in range(10)]}")

# Construct training and test sets
dataset = []
for digit, imgs in image_map.items():
    for img in imgs:
        # Create 1-hot target vector
        y = np.zeros((10, 1))
        y[digit] = 1.0
        dataset.append((img.reshape(784, 1), y))

random.seed(42)
random.shuffle(dataset)

split_idx = int(0.8 * len(dataset))
train_set = dataset[:split_idx]
test_set = dataset[split_idx:]

print(f"Dataset split: {len(train_set)} training samples, {len(test_set)} test samples.")

Loaded sample images for digits: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Number of samples per digit: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Dataset split: 80 training samples, 20 test samples.


### Training a Fresh Network with Mini-Batch SGD
Let's initialize a new network $[784 \to 16 \to 16 \to 10]$ and train it for 30 epochs:

In [8]:
# Ensure Network class is defined (fallback if previous cells were skipped)
import os
import sys
import pickle
import random
import numpy as np

if 'Network' not in globals():
    try:
        from neural_network_core import Network, sigmoid, sigmoid_prime
    except ImportError:
        pass

# Ensure test_set and train_set are available
if 'train_set' not in globals() or 'test_set' not in globals():
    image_map_path = os.path.join("_2017", "nn", "image_map")
    with open(image_map_path, "rb") as f:
        image_map = pickle.load(f, encoding="latin1")
    dataset = []
    for digit, imgs in image_map.items():
        for img in imgs:
            y = np.zeros((10, 1))
            y[digit] = 1.0
            dataset.append((img.reshape(784, 1), y))
    random.seed(42)
    random.shuffle(dataset)
    split_idx = int(0.8 * len(dataset))
    train_set = dataset[:split_idx]
    test_set = dataset[split_idx:]

# Initialize fresh network
net = Network([784, 16, 16, 10])

print("Initial untrained performance:")
initial_acc = net.evaluate(test_set) / len(test_set) * 100
print(f"Untrained Accuracy: {initial_acc:.1f}% (random chance is ~10%)\n")

print("Starting Stochastic Gradient Descent training...")
history = net.SGD(
    training_data=train_set,
    epochs=30,
    mini_batch_size=10,
    eta=3.0,
    test_data=test_set
)

Initial untrained performance:
Untrained Accuracy: 5.0% (random chance is ~10%)

Starting Stochastic Gradient Descent training...
Epoch  1 / 30: 1 / 20 (5.00%) correct
Epoch  2 / 30: 1 / 20 (5.00%) correct
Epoch  3 / 30: 1 / 20 (5.00%) correct
Epoch  4 / 30: 2 / 20 (10.00%) correct
Epoch  5 / 30: 0 / 20 (0.00%) correct
Epoch  6 / 30: 0 / 20 (0.00%) correct
Epoch  7 / 30: 1 / 20 (5.00%) correct
Epoch  8 / 30: 3 / 20 (15.00%) correct
Epoch  9 / 30: 0 / 20 (0.00%) correct
Epoch 10 / 30: 0 / 20 (0.00%) correct
Epoch 11 / 30: 0 / 20 (0.00%) correct
Epoch 12 / 30: 0 / 20 (0.00%) correct
Epoch 13 / 30: 3 / 20 (15.00%) correct
Epoch 14 / 30: 3 / 20 (15.00%) correct
Epoch 15 / 30: 3 / 20 (15.00%) correct
Epoch 16 / 30: 5 / 20 (25.00%) correct
Epoch 17 / 30: 3 / 20 (15.00%) correct
Epoch 18 / 30: 5 / 20 (25.00%) correct
Epoch 19 / 30: 5 / 20 (25.00%) correct
Epoch 20 / 30: 3 / 20 (15.00%) correct
Epoch 21 / 30: 8 / 20 (40.00%) correct
Epoch 22 / 30: 5 / 20 (25.00%) correct
Epoch 23 / 30: 10 / 20

### Loading the Pretrained Network
We can load a fully trained $[784 \to 16 \to 16 \to 10]$ network trained on the MNIST training images. Let's load the pretrained weights and evaluate!

In [9]:
# Ensure Network class and dataset are defined
import os
import sys
import pickle
import random
import numpy as np

if 'Network' not in globals():
    try:
        from neural_network_core import Network
    except ImportError:
        pass

if 'dataset' not in globals():
    image_map_path = os.path.join("_2017", "nn", "image_map")
    with open(image_map_path, "rb") as f:
        image_map = pickle.load(f, encoding="latin1")
    dataset = []
    for digit, imgs in image_map.items():
        for img in imgs:
            y = np.zeros((10, 1))
            y[digit] = 1.0
            dataset.append((img.reshape(784, 1), y))

# Load pretrained network
pretrained_net = Network([784, 16, 16, 10])
pretrained_path = os.path.join("_2017", "nn", "pretrained_weights_and_biases")

with open(pretrained_path, "rb") as f:
    weights, biases = pickle.load(f, encoding="latin1")
    pretrained_net.weights = weights
    pretrained_net.biases = biases

pretrained_correct = pretrained_net.evaluate(dataset)
print(f"Pretrained Network Accuracy on sample dataset: {pretrained_correct} / {len(dataset)} ({pretrained_correct/len(dataset)*100:.1f}%)")

Pretrained Network Accuracy on sample dataset: 96 / 100 (96.0%)


### Interactive Digit Visualizer & Probability Distribution
Let's visualize a digit from the dataset, display its $28 \times 28$ grayscale pixel layout, and inspect the network's output activations!

In [10]:
# Ensure pretrained_net and image_map are available
import os
import sys
import pickle
import numpy as np

if 'pretrained_net' not in globals():
    if 'Network' not in globals():
        try:
            from neural_network_core import Network
        except ImportError:
            pass
    pretrained_net = Network([784, 16, 16, 10])
    with open(os.path.join("_2017", "nn", "pretrained_weights_and_biases"), "rb") as f:
        w, b = pickle.load(f, encoding="latin1")
        pretrained_net.weights = w
        pretrained_net.biases = b

if 'image_map' not in globals():
    with open(os.path.join("_2017", "nn", "image_map"), "rb") as f:
        image_map = pickle.load(f, encoding="latin1")

def display_digit_and_prediction(network, digit_image, true_label=None):
    """Displays ASCII rendering of 28x28 digit and horizontal probability bar chart."""
    img_2d = digit_image.reshape(28, 28)
    
    # 1. ASCII Art Representation
    print("━" * 44)
    if true_label is not None:
        print(f" 🖼️ Input Image (True Label: {true_label})")
    print("━" * 44)
    for row in range(0, 28, 2):  # Skip alternate rows for square aspect ratio in terminal
        line = ""
        for col in range(28):
            val = img_2d[row, col]
            if val > 0.7:
                line += "██"
            elif val > 0.3:
                line += "▒▒"
            elif val > 0.05:
                line += "░░"
            else:
                line += "  "
        print(f"  {line}")
    print("━" * 44)
    
    # 2. Forward pass prediction
    output = network.feedforward(digit_image.reshape(784, 1)).flatten()
    prediction = np.argmax(output)
    
    print(f" 🎯 Network Prediction: {prediction} (Confidence: {output[prediction]*100:.1f}%)")
    print(" 📊 Output Layer Activations:")
    for digit in range(10):
        bar_len = int(output[digit] * 30)
        bar = "█" * bar_len + "░" * (30 - bar_len)
        star = " 🌟" if digit == prediction else ""
        print(f"   [{digit}]: {bar}  {output[digit]:.3f}{star}")
    print("━" * 44)

# Test on a sample digit 7
sample_digit = 7
sample_img = image_map[sample_digit][0]

display_digit_and_prediction(pretrained_net, sample_img, true_label=sample_digit)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 🖼️ Input Image (True Label: 7)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                                          
                                                          
                                                          
                                                          
                          ░░▒▒▒▒██████████████            
                ░░████████████░░░░░░░░████████            
              ░░████████░░          ▒▒████▒▒              
              ▒▒████░░          ▒▒██████░░                
                              ██████▒▒                    
                            ██████▒▒                      
                      ░░▒▒████▒▒                          
                    ░░▒▒████░░                            
                  ░░▒▒██▒▒░░                              
                  ░░██▒▒                                  
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 🎯 Netw

---
## 🎓 Summary & Key Takeaways

1. **Neurons** are holders of numbers ($a \in [0, 1]$).
2. **Forward Propagation** computes activations layer-by-layer:
   $$\mathbf{a}^{(l)} = \sigma\left( \mathbf{W}^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)} \right)$$
3. **The Cost Function** $C(W, b)$ measures the distance between the network's predictions and ground truth.
4. **Gradient Descent** moves iteratively in the direction of $-\nabla C$ to minimize error.
5. **Backpropagation** applies the Chain Rule backwards from output to input to calculate $\frac{\partial C}{\partial W}$ and $\frac{\partial C}{\partial b}$ efficiently.

---
### 🚀 Next Steps
- Try modifying the network architecture (e.g., test $[784, 32, 10]$ or $[784, 64, 32, 10]$).
- Experiment with different learning rates $\eta$ and mini-batch sizes.
- Try implementing the **ReLU** activation function and cross-entropy loss!